# Market A Simulation
Cases: full fill only → partial fill only → both → forced hedge → full day

In [1]:
import sys, copy
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append('../src')

import utils.stock_simulation as stock_mod
import utils.market_simulator as market_mod
import utils.order_book.order_book_impl as book_mod
import utils.order_book.graphic_utils as graph_utils
import utils.market_maker.quoter as quoter_mod
import utils.client_flow.flow_generator as flow_mod

for mod in [stock_mod, market_mod, book_mod, graph_utils, quoter_mod, flow_mod]:
    importlib.reload(mod)

from utils.stock_simulation import Stock
from utils.market_simulator import Market
from utils.order_book.order_book_impl import Order_book, Order, generate_order_id
from utils.order_book.graphic_utils import plot_order_book
from utils.market_maker.quoter import Quoter, QuoterConfig
from utils.client_flow.flow_generator import ClientFlowGenerator

SEED = 42
np.random.seed(SEED)

In [2]:
stock = Stock(drift=0.0, vol=0.20)
stock.simulate_garch(n_days=1, dt_seconds=0.01, alpha=0.05, beta=0.94, lam=756, sigma_J=0.005)

market_B = Market(stock=stock)
market_B.generate_noised_mid_price()
market_B.build_spread(option="Skew", window_size=600, alpha=0.5, gamma=0.3, ema_span=500, threshold=3)

market_C = copy.deepcopy(market_B)
market_C.build_spread(option="Adaptive", window_size=600)

dt = stock.time_step
CAPITAL = 1_000_000.0
cfg = QuoterConfig(requote_threshold_spread_fraction=0.25)

def new_sim():
    book = Order_book()
    mm = Quoter(market_B, market_C, config=cfg, capital_K=CAPITAL)
    book.register_quoter_listener(mm.on_fill)
    return book, mm

def post_initial_ladder(book, mm, step=0):
    book.tick(step)
    quotes, cancels = mm.compute_quotes(step, step * dt, book.mm_resting_orders)
    book.cancel_orders(cancels)
    book.post_mm_quotes(quotes)
    print(f'  Ladder posted: {len(quotes)} orders')

def show_history(mm):
    df = mm.trade_history
    cols = ['step', 'direction', 'level', 'price', 'size', 'is_full_fill',
            'cash_flow', 'inventory_after', 'is_hedge', 'venue']
    print(df[cols].to_string(index=False))
    print(f'\nInventory: {mm.inventory:.0f} EUR')

Coefficient annualization : 46661.333028536596


## Case 1 — Full fill only
Client buys the entire level-1 ask → slot gone → re-quoted at next step.

In [7]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

resting = book.mm_resting_orders
ask_l1_price, ask_l1_id = min((v['price'], oid) for oid, v in resting.items() if v['direction'] == 'sell')
ask_l1_size = resting[ask_l1_id]['original_size']
print(f'  Level-1 ask: price={ask_l1_price:.5f}, size={ask_l1_size}')

book.route_client_order(Order(generate_order_id(), 'buy', ask_l1_price, ask_l1_size, 'limit_order', 'client'))
print(f'  After fill — _pending_fills: {len(mm._pending_fills)}, _pending_topups: {len(mm._pending_topups)}')

book.tick(1)
quotes, cancels = mm.compute_quotes(1, 1 * dt, book.mm_resting_orders)
book.cancel_orders(cancels)
book.post_mm_quotes(quotes)
print(f'  Requote step: {len(quotes)} new order(s), {len(cancels)} cancel(s)\n')
show_history(mm)

  Ladder posted: 20 orders
  Level-1 ask: price=100.00670, size=74082
  After fill — _pending_fills: 1, _pending_topups: 0
  Requote step: 20 new order(s), 19 cancel(s)

 step direction  level    price  size  is_full_fill    cash_flow  inventory_after  is_hedge venue
    0      sell      1 100.0067 74082          True 7.407955e+06         -74082.0     False     A

Inventory: -74082 EUR


## Case 2 — Partial fill only
Client sells half of the level-1 bid → order still resting at reduced size → top-up posted.

In [8]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

resting = book.mm_resting_orders
bid_l1_price, bid_l1_id = max((v['price'], oid) for oid, v in resting.items() if v['direction'] == 'buy')
bid_l1_size = resting[bid_l1_id]['original_size']
partial_size = bid_l1_size // 2
print(f'  Level-1 bid: price={bid_l1_price:.5f}, size={bid_l1_size}')
print(f'  Sending partial sell: {partial_size}')

book.route_client_order(Order(generate_order_id(), 'sell', bid_l1_price, partial_size, 'limit_order', 'client'))
remaining = book.mm_resting_orders.get(bid_l1_id, {}).get('remaining_size', 'gone')
print(f'  Remaining on that order: {remaining}  (original: {bid_l1_size})')
print(f'  After fill — _pending_fills: {len(mm._pending_fills)}, _pending_topups: {len(mm._pending_topups)}')

book.tick(1)
quotes, cancels = mm.compute_quotes(1, 1 * dt, book.mm_resting_orders)
book.cancel_orders(cancels)
book.post_mm_quotes(quotes)
print(f'  Top-up step: {len(quotes)} new order(s), {len(cancels)} cancel(s)\n')
show_history(mm)

  Ladder posted: 20 orders
  Level-1 bid: price=99.99330, size=74082
  Sending partial sell: 37041
  Remaining on that order: 37041  (original: 74082)
  After fill — _pending_fills: 0, _pending_topups: 1
  Top-up step: 20 new order(s), 20 cancel(s)

 step direction  level   price  size  is_full_fill     cash_flow  inventory_after  is_hedge venue
    0       buy      1 99.9933 37041         False -3.704222e+06          37041.0     False     A

Inventory: 37041 EUR


## Case 3 — Full fill + partial fill in the same step
Two client orders arrive: one fully consumes ask L1, one partially fills bid L1.
Both handled in one Priority-4 pass: re-quote + top-up emitted together.

In [9]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

resting = book.mm_resting_orders
ask_l1_price, ask_l1_id = min((v['price'], oid) for oid, v in resting.items() if v['direction'] == 'sell')
ask_l1_size = resting[ask_l1_id]['original_size']
bid_l1_price, bid_l1_id = max((v['price'], oid) for oid, v in resting.items() if v['direction'] == 'buy')
bid_l1_size = resting[bid_l1_id]['original_size']
partial_size = bid_l1_size // 2

print(f'  Full fill  → client buy  {ask_l1_size} @ {ask_l1_price:.5f} (ask L1)')
print(f'  Partial    → client sell {partial_size} @ {bid_l1_price:.5f} (bid L1, half size)')

book.route_client_order(Order(generate_order_id(), 'buy',  ask_l1_price, ask_l1_size,  'limit_order', 'client'))
book.route_client_order(Order(generate_order_id(), 'sell', bid_l1_price, partial_size, 'limit_order', 'client'))
print(f'  After fills — _pending_fills: {len(mm._pending_fills)}, _pending_topups: {len(mm._pending_topups)}')

book.tick(1)
quotes, cancels = mm.compute_quotes(1, 1 * dt, book.mm_resting_orders)
book.cancel_orders(cancels)
book.post_mm_quotes(quotes)
print(f'  Requote + top-up: {len(quotes)} new order(s), {len(cancels)} cancel(s)\n')
show_history(mm)

  Ladder posted: 20 orders
  Full fill  → client buy  74082 @ 100.00670 (ask L1)
  Partial    → client sell 37041 @ 99.99330 (bid L1, half size)
  After fills — _pending_fills: 1, _pending_topups: 1
  Requote + top-up: 20 new order(s), 19 cancel(s)

 step direction  level    price  size  is_full_fill     cash_flow  inventory_after  is_hedge venue
    0      sell      1 100.0067 74082          True  7.407955e+06         -74082.0     False     A
    0       buy      1  99.9933 37041         False -3.704222e+06         -37041.0     False     A

Inventory: -37041 EUR


## Case 4 — Forced hedge
Inventory pre-loaded above 90% → hedge fires, returns to flat (50% EUR / 50% USD).

In [10]:
book, mm = new_sim()
post_initial_ladder(book, mm, step=0)

mm.inventory = CAPITAL * 0.91
print(f'  Inventory pre-loaded: {mm.inventory:.0f} EUR ({mm.inventory/CAPITAL:.0%} of capital)')
print(f'  needs_hedge: {mm.needs_hedge()}')

resting = book.mm_resting_orders
bids = [v['price'] for v in resting.values() if v['direction'] == 'buy']
asks = [v['price'] for v in resting.values() if v['direction'] == 'sell']
mid = (max(bids) + min(asks)) / 2

hedged = mm.execute_hedge(step=0, t=0.0, fair_mid=mid)
print(f'  Hedge executed: {hedged}')
print(f'  Inventory after: {mm.inventory:.2f} EUR ({mm.inventory/CAPITAL:.2%}) → flat = 50/50\n')
show_history(mm)

  Ladder posted: 20 orders
  Inventory pre-loaded: 910000 EUR (91% of capital)
  needs_hedge: True
  Hedge executed: True
  Inventory after: 0.00 EUR (0.00%) → flat = 50/50

 step direction  level  price      size  is_full_fill    cash_flow  inventory_after  is_hedge venue
    0      sell      0  100.0 545996.36          True 5.458872e+07        364003.64      True     B
    0      sell      0  100.0 364003.64          True 3.638944e+07             0.00      True     C

Inventory: 0 EUR


## Full day simulation

In [ ]:
book2, mm2 = new_sim()
gen = ClientFlowGenerator(seed=SEED)
N = stock.n_steps

for step in range(N):
    t = step * dt
    book2.tick(step)

    quotes, cancels = mm2.compute_quotes(step, t, book2.mm_resting_orders)
    book2.cancel_orders(cancels)
    book2.post_mm_quotes(quotes)

    resting = book2.mm_resting_orders
    bids = [v['price'] for v in resting.values() if v['direction'] == 'buy']
    asks = [v['price'] for v in resting.values() if v['direction'] == 'sell']
    if not bids or not asks:
        continue

    best_bid, best_ask = max(bids), min(asks)
    mid = (best_bid + best_ask) / 2

    for order in gen.generate_step(mid_price=mid, best_bid=best_bid, best_ask=best_ask, dt=dt):
        book2.route_client_order(order)

    mm2.execute_hedge(step, t, mid)

df = mm2.trade_history
mm_fills = df[~df['is_hedge']]
hedges   = df[df['is_hedge']]
print(f'Total trades : {len(df)}  ({len(mm_fills)} MM fills, {len(hedges)} hedge legs)')
print(f'Final inventory : {mm2.inventory:.0f} EUR')

In [ ]:
df['cum_cash'] = df['cash_flow'].cumsum()
df['cum_fees'] = df['fee_cost'].cumsum()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.patch.set_facecolor('#111111')
for ax in axes:
    ax.set_facecolor('#111111')
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='#444444')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#444444')
    ax.spines['bottom'].set_color('#444444')

axes[0].plot(df['t'], df['cum_cash'], color='#00ff88', linewidth=0.8, label='Total')
if len(hedges):
    axes[0].scatter(hedges['t'], df.loc[hedges.index, 'cum_cash'],
                    color='#ff4444', s=12, zorder=5, label='Hedge')
axes[0].axhline(0, color='#444', linewidth=0.6)
axes[0].set_title('Cumulative cash P&L', color='white', fontsize=13)
axes[0].set_ylabel('P&L', color='white')
axes[0].legend(facecolor='#222222', edgecolor='#444444', labelcolor='white', fontsize=9)

axes[1].plot(df['t'], df['inventory_after'], color='#ff9500', linewidth=0.8)
axes[1].axhline(0, color='#444', linewidth=0.6)
lim = mm2.capital_K * mm2.cfg.delta_limit
axes[1].axhline( lim, color='#ff4444', linewidth=0.6, linestyle='--', label='+90% limit')
axes[1].axhline(-lim, color='#ff4444', linewidth=0.6, linestyle='--', label='-90% limit')
axes[1].set_title('Inventory (EUR)', color='white', fontsize=13)
axes[1].set_ylabel('EUR', color='white')
axes[1].legend(facecolor='#222222', edgecolor='#444444', labelcolor='white', fontsize=9)

axes[2].plot(df['t'], df['cum_fees'], color='#ff4444', linewidth=0.8)
axes[2].set_title('Cumulative fees', color='white', fontsize=13)
axes[2].set_ylabel('Fees', color='white')
axes[2].set_xlabel('Time (s)', color='white')

plt.tight_layout()
plt.show()

print(f'Net cash P&L   : {df["cash_flow"].sum():.2f}')
print(f'Total fees     : {df["fee_cost"].sum():.2f}')
print(f'  MM maker fees: {mm_fills["fee_cost"].sum():.2f}')
print(f'  Hedge fees   : {hedges["fee_cost"].sum():.2f}')